# 17 — Chunking Specifies Large Model Distribution

**Leading specification:** large artifacts become distributable when they are split into independently verifiable transfer units.

Notebook 07 established that content identity follows bytes:

\[
I = H(C)
\]

Notebook 13 established that trust follows local verification:

\[
V=\bigl(H(C_{\mathrm{received}})=H(C_{\mathrm{published}})\bigr)
\]

Notebook 17 extends both ideas to large model artifacts. Instead of moving one monolithic file, the artifact is split into chunks. Each chunk has its own digest. Reassembly succeeds only when every required chunk verifies.

## 1. Context

Large model checkpoints are often too large to treat as a single fragile transfer.

Chunking changes the engineering object from:

```text
one large file
```

to:

```text
many independently verifiable transfer units
```

This notebook develops the minimal chunking workflow:

```text
artifact
→ fixed-size chunks
→ chunk digests
→ chunk manifest
→ verify chunks
→ reassemble artifact
```

Notebook 17 does not implement a production protocol. It specifies the mechanics that later notebooks can use for replication, peer-to-peer transfer, and reproducibility.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import textwrap
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Robust paths whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "17_chunking"
SITE = REPO_ROOT / "site"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
except Exception:
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")

## 2. Chunking variables

The artifact content \(C\) is represented as a sequence of chunks:

\[
C = (c_0,c_1,\ldots,c_{n-1})
\]

Each chunk receives an identity:

\[
I_i = H(c_i)
\]

The chunk manifest records all required chunk identities:

\[
M=\{(i, I_i, |c_i|)\}_{i=0}^{n-1}
\]

Reassembly succeeds only when every chunk verifies.

In [ ]:
@dataclass(frozen=True)
class ChunkingVariables:
    artifact: str
    chunk: str
    chunk_identity: str
    manifest: str
    reassembly: str

variables = ChunkingVariables(
    artifact="C = full model/software/data artifact",
    chunk="c_i = independently movable transfer unit",
    chunk_identity="I_i = H(c_i)",
    manifest="M = ordered list of chunk indices, sizes, and digests",
    reassembly="C is reconstructed only after all chunks verify",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})

## 3. Create a toy large artifact

The artifact is deterministic so the notebook is reproducible. The exact bytes matter: if one chunk changes, the corresponding digest changes.

In [ ]:
artifact_dir = RESULTS / "artifacts"
chunk_dir = RESULTS / "chunks"
received_dir = RESULTS / "received_chunks"
reassembly_dir = RESULTS / "reassembly"

for directory in [artifact_dir, chunk_dir, received_dir, reassembly_dir]:
    directory.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / "toy-large-model.bin"

payload = (
    b"open-model-distribution chunking notebook\n"
    + b"toy large model checkpoint\n"
    + bytes(range(256)) * 512
    + b"\nmanifest-ready artifact\n"
)

artifact_path.write_bytes(payload)

artifact_digest = sha256_file(artifact_path)
artifact_cid = content_id(artifact_path)

artifact_record = {
    "artifact": str(artifact_path.relative_to(REPO_ROOT)),
    "size_bytes": artifact_path.stat().st_size,
    "algorithm": "sha256",
    "digest": artifact_digest,
    "content_id": artifact_cid,
}

artifact_record_path = RESULTS / "17_artifact_identity.json"
artifact_record_path.write_text(json.dumps(artifact_record, indent=2), encoding="utf-8")

artifact_record

## 4. Split the artifact into chunks

Notebook 17 uses fixed-size chunks to keep the specification simple.

Production systems can use different chunking rules, but the verification pattern is the same: each chunk must be independently identified and checked.

In [ ]:
def chunk_bytes(data: bytes, chunk_size: int):
    chunks = []
    for index, start in enumerate(range(0, len(data), chunk_size)):
        chunk = data[start:start + chunk_size]
        chunks.append({
            "index": index,
            "offset_start": start,
            "offset_end": start + len(chunk),
            "size_bytes": len(chunk),
            "sha256": sha256_bytes(chunk),
            "content_id": f"sha256:{sha256_bytes(chunk)}",
        })
    return chunks

CHUNK_SIZE = 2048
data = artifact_path.read_bytes()
chunks = chunk_bytes(data, CHUNK_SIZE)

# Write chunks as separate transfer units.
for row in chunks:
    chunk_path = chunk_dir / f"chunk_{row['index']:04d}.bin"
    start = row["offset_start"]
    end = row["offset_end"]
    chunk_path.write_bytes(data[start:end])
    row["path"] = str(chunk_path.relative_to(REPO_ROOT))

chunk_df = pd.DataFrame(chunks)
chunk_table_csv = RESULTS / "17_chunk_table.csv"
chunk_df.to_csv(chunk_table_csv, index=False)

chunk_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axis("off")

ax.set_title("Chunking pipeline", fontsize=17, pad=18)

# Left artifact
ax.text(0.08, 0.55, "model.bin", ha="center", va="center", fontsize=14,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes)

ax.annotate("", xy=(0.22, 0.55), xytext=(0.14, 0.55),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

# Chunk boxes
x0 = 0.25
w = 0.07
for i in range(5):
    label = str(i) if i < 4 else "…"
    ax.add_patch(plt.Rectangle((x0 + i * (w + 0.02), 0.43), w, 0.24,
                               fill=False, lw=1.8, transform=ax.transAxes))
    ax.text(x0 + i * (w + 0.02) + w / 2, 0.55, label,
            ha="center", va="center", fontsize=13, transform=ax.transAxes)

ax.text(0.43, 0.34, f"{CHUNK_SIZE}-byte transfer units", ha="center", fontsize=12, transform=ax.transAxes)

ax.annotate("", xy=(0.66, 0.55), xytext=(0.58, 0.55),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.73, 0.55, "hash each\nchunk", ha="center", va="center", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes)

ax.annotate("", xy=(0.86, 0.55), xytext=(0.80, 0.55),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.92, 0.55, "chunk\nmanifest", ha="center", va="center", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes)

fig.tight_layout()
chunking_pipeline_fig = FIGURES / "17_chunking_pipeline.png"
fig.savefig(chunking_pipeline_fig, dpi=180)
plt.show()

chunking_pipeline_fig

## 5. Build the chunk manifest

The manifest records the ordered chunk identities needed to reconstruct the artifact.

In [ ]:
chunk_manifest = {
    "schema": "open-model-distribution.chunk-manifest.v0",
    "artifact": str(artifact_path.relative_to(REPO_ROOT)),
    "artifact_size_bytes": artifact_path.stat().st_size,
    "artifact_algorithm": "sha256",
    "artifact_digest": artifact_digest,
    "artifact_content_id": artifact_cid,
    "chunk_size_bytes": CHUNK_SIZE,
    "chunk_count": len(chunks),
    "chunks": [
        {
            "index": int(row["index"]),
            "offset_start": int(row["offset_start"]),
            "offset_end": int(row["offset_end"]),
            "size_bytes": int(row["size_bytes"]),
            "sha256": row["sha256"],
            "content_id": row["content_id"],
            "path": row["path"],
        }
        for _, row in chunk_df.iterrows()
    ],
}

chunk_manifest_path = RESULTS / "17_chunk_manifest.json"
chunk_manifest_path.write_text(json.dumps(chunk_manifest, indent=2), encoding="utf-8")

{
    "chunk_count": chunk_manifest["chunk_count"],
    "chunk_size_bytes": chunk_manifest["chunk_size_bytes"],
    "artifact_digest_prefix": artifact_digest[:16] + "…",
}

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.axis("off")
ax.set_title("Chunk manifest", fontsize=17, pad=16)

ax.text(0.18, 0.78, "artifact digest", ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
ax.text(0.22, 0.78, artifact_digest[:24] + "…", ha="left", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.0),
        transform=ax.transAxes)

ax.text(0.18, 0.66, "chunk size", ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
ax.text(0.22, 0.66, f"{CHUNK_SIZE} bytes", ha="left", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.0),
        transform=ax.transAxes)

ax.text(0.18, 0.54, "chunk count", ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
ax.text(0.22, 0.54, str(len(chunks)), ha="left", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.0),
        transform=ax.transAxes)

ax.text(0.62, 0.78, "ordered chunk identities", ha="center", va="center", fontsize=13, weight="bold", transform=ax.transAxes)

preview_rows = chunk_df.head(5)
y = 0.66
for _, row in preview_rows.iterrows():
    ax.text(0.46, y, f"{int(row['index'])}", ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", lw=1.0), transform=ax.transAxes)
    ax.text(0.53, y, row["sha256"][:22] + "…", ha="left", va="center", fontsize=10, transform=ax.transAxes)
    y -= 0.10

if len(chunk_df) > 5:
    ax.text(0.62, y, "…", ha="center", va="center", fontsize=14, transform=ax.transAxes)

fig.tight_layout()
chunk_manifest_fig = FIGURES / "17_chunk_manifest.png"
fig.savefig(chunk_manifest_fig, dpi=180)
plt.show()

chunk_manifest_fig

## 6. Verify chunks independently

Each received chunk is accepted only if its digest matches the manifest.

\[
V_i=\bigl(H(c_i^{\mathrm{received}})=I_i\bigr)
\]

In [ ]:
# Simulate receiving all chunks.
for _, row in chunk_df.iterrows():
    src = REPO_ROOT / row["path"]
    dst = received_dir / f"chunk_{int(row['index']):04d}.bin"
    shutil.copyfile(src, dst)

def verify_chunks(received_directory: Path, manifest: dict):
    records = []
    for chunk in manifest["chunks"]:
        index = chunk["index"]
        expected = chunk["sha256"]
        path = received_directory / f"chunk_{index:04d}.bin"
        exists = path.exists()
        local = sha256_file(path) if exists else ""
        passed = exists and local == expected
        records.append({
            "index": index,
            "path": str(path.relative_to(REPO_ROOT)) if path.exists() else str(path),
            "exists": exists,
            "expected_sha256": expected,
            "local_sha256": local,
            "verification": "PASS" if passed else "FAIL",
            "accepted": passed,
        })
    return pd.DataFrame(records)

verification_df = verify_chunks(received_dir, chunk_manifest)
verification_csv = RESULTS / "17_chunk_verification.csv"
verification_df.to_csv(verification_csv, index=False)

verification_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
values = verification_df["accepted"].astype(int)
ax.bar(verification_df["index"], values)
ax.set_ylim(0, 1.2)
ax.set_xlabel("chunk index")
ax.set_ylabel("verification")
ax.set_yticks([0, 1])
ax.set_yticklabels(["FAIL", "PASS"])
ax.set_title("Independent chunk verification")
fig.tight_layout()

chunk_verification_fig = FIGURES / "17_chunk_verification.png"
fig.savefig(chunk_verification_fig, dpi=180)
plt.show()

chunk_verification_fig

## 7. Reassemble only when every chunk verifies

Reassembly is a gated operation:

\[
R_{\mathrm{assemble}}=\bigwedge_i V_i
\]

If any chunk fails, the reconstructed artifact should not be accepted.

In [ ]:
def reassemble_chunks(received_directory: Path, manifest: dict, output_path: Path):
    verification = verify_chunks(received_directory, manifest)
    if not verification["accepted"].all():
        return {
            "reassembled": False,
            "reason": "one or more chunks failed verification",
            "failed_indices": verification.loc[~verification["accepted"], "index"].tolist(),
            "output": None,
        }

    with output_path.open("wb") as handle:
        for chunk in manifest["chunks"]:
            path = received_directory / f"chunk_{chunk['index']:04d}.bin"
            handle.write(path.read_bytes())

    digest = sha256_file(output_path)
    return {
        "reassembled": True,
        "output": str(output_path.relative_to(REPO_ROOT)),
        "digest": digest,
        "matches_original": digest == manifest["artifact_digest"],
        "failed_indices": [],
    }

reassembled_path = reassembly_dir / "reassembled-model.bin"
reassembly_report = reassemble_chunks(received_dir, chunk_manifest, reassembled_path)

reassembly_report_path = RESULTS / "17_reassembly_report.json"
reassembly_report_path.write_text(json.dumps(reassembly_report, indent=2), encoding="utf-8")

reassembly_report

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.axis("off")

nodes = [
    ("verified\nchunks", 0.18),
    ("ordered\nreassembly", 0.50),
    ("artifact\ndigest check", 0.82),
]

for label, x in nodes:
    ax.text(x, 0.60, label, ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round,pad=0.42", fc="white", ec="black", lw=1.5),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(x2 - 0.11, 0.60), xytext=(x1 + 0.11, 0.60),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

status = "PASS" if reassembly_report["matches_original"] else "FAIL"
ax.text(0.5, 0.25, f"reassembled artifact digest check: {status}", ha="center", fontsize=13, transform=ax.transAxes)

ax.set_title("Reassembly is gated by chunk verification", fontsize=16, pad=16)
fig.tight_layout()

reassembly_fig = FIGURES / "17_reassembly.png"
fig.savefig(reassembly_fig, dpi=180)
plt.show()

reassembly_fig

## 8. Detect a corrupted chunk

Now corrupt one received chunk and verify again. The manifest identifies the failing transfer unit.

In [ ]:
corrupt_dir = RESULTS / "received_chunks_corrupt"
if corrupt_dir.exists():
    shutil.rmtree(corrupt_dir)
shutil.copytree(received_dir, corrupt_dir)

corrupt_index = 2 if len(chunks) > 2 else 0
corrupt_path = corrupt_dir / f"chunk_{corrupt_index:04d}.bin"

corrupt_bytes = bytearray(corrupt_path.read_bytes())
corrupt_bytes[10] = (corrupt_bytes[10] + 1) % 256
corrupt_path.write_bytes(bytes(corrupt_bytes))

corrupt_verification_df = verify_chunks(corrupt_dir, chunk_manifest)
corrupt_report_csv = RESULTS / "17_corrupt_chunk_report.csv"
corrupt_verification_df.to_csv(corrupt_report_csv, index=False)

failed = corrupt_verification_df[~corrupt_verification_df["accepted"]]
failed[["index", "verification", "expected_sha256", "local_sha256"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))

values = corrupt_verification_df["accepted"].astype(int)
ax.bar(corrupt_verification_df["index"], values)
ax.set_ylim(0, 1.2)
ax.set_xlabel("chunk index")
ax.set_ylabel("verification")
ax.set_yticks([0, 1])
ax.set_yticklabels(["FAIL", "PASS"])
ax.set_title("Corrupt chunk detection")

for _, row in corrupt_verification_df[~corrupt_verification_df["accepted"]].iterrows():
    ax.text(row["index"], 0.08, "corrupt", ha="center", va="bottom", fontsize=10)

fig.tight_layout()
corrupt_chunk_fig = FIGURES / "17_corrupt_chunk_detection.png"
fig.savefig(corrupt_chunk_fig, dpi=180)
plt.show()

corrupt_chunk_fig

## 9. Preview: Merkle-style aggregation

Chunk identities can be aggregated into a higher-level identity. Notebook 17 only previews this idea.

A simple binary tree combines adjacent chunk hashes until one root digest remains. Later notebooks can use this as a bridge to reproducible releases and distributed object graphs.

In [ ]:
def merkle_level(hex_hashes):
    out = []
    for i in range(0, len(hex_hashes), 2):
        left = bytes.fromhex(hex_hashes[i])
        right = bytes.fromhex(hex_hashes[i + 1]) if i + 1 < len(hex_hashes) else left
        out.append(hashlib.sha256(left + right).hexdigest())
    return out

levels = [chunk_df["sha256"].tolist()]
while len(levels[-1]) > 1:
    levels.append(merkle_level(levels[-1]))

merkle_root = levels[-1][0]
merkle_preview = {
    "chunk_count": len(chunks),
    "level_counts": [len(level) for level in levels],
    "merkle_root_preview": merkle_root,
}

merkle_preview_path = RESULTS / "17_merkle_preview.json"
merkle_preview_path.write_text(json.dumps(merkle_preview, indent=2), encoding="utf-8")

merkle_preview

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.axis("off")

ax.set_title("Merkle-style aggregation preview", fontsize=16, pad=16)

level_counts = merkle_preview["level_counts"][:4]
y_positions = [0.20, 0.40, 0.60, 0.78]

for level_idx, count in enumerate(level_counts):
    y = y_positions[level_idx]
    shown = min(count, 8)
    xs = [0.15 + i * (0.70 / max(shown - 1, 1)) for i in range(shown)]
    for x in xs:
        ax.text(x, y, "H", ha="center", va="center", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="black", lw=1.0),
                transform=ax.transAxes)
    ax.text(0.90, y, f"level {level_idx}: {count}", ha="left", va="center", fontsize=10, transform=ax.transAxes)

ax.annotate("hash pairs upward", xy=(0.50, 0.72), xytext=(0.50, 0.29),
            arrowprops=dict(arrowstyle="->", lw=2), ha="center", xycoords=ax.transAxes)

ax.text(0.5, 0.08, "root: " + merkle_root[:18] + "…", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
merkle_preview_fig = FIGURES / "17_merkle_preview.png"
fig.savefig(merkle_preview_fig, dpi=180)
plt.show()

merkle_preview_fig

## 10. Engineering summary

| Property | Specification | Notebook role |
|---|---|---|
| Transfer unit | Fixed-size chunk | Large artifacts become movable pieces. |
| Chunk identity | \(I_i=H(c_i)\) | Each transfer unit verifies independently. |
| Manifest | Ordered chunk list | Reassembly has a canonical recipe. |
| Reassembly | \(\bigwedge_i V_i\) | Artifact reconstruction is gated by verification. |
| Corruption detection | Failed chunk digest | The failing unit is isolated by index. |

## 11. Engineering statement

Chunking specifies large model distribution by turning one large artifact into independently verifiable transfer units. Each chunk has a digest, the manifest records chunk order and identity, and reassembly succeeds only when every required chunk verifies.

## 12. Key equations

Artifact as chunks:

\[
C=(c_0,c_1,\ldots,c_{n-1})
\]

Chunk identity:

\[
I_i=H(c_i)
\]

Chunk manifest:

\[
M=\{(i,I_i,|c_i|)\}_{i=0}^{n-1}
\]

Chunk verification:

\[
V_i=\bigl(H(c_i^{\mathrm{received}})=I_i\bigr)
\]

Reassembly gate:

\[
R_{\mathrm{assemble}}=\bigwedge_{i=0}^{n-1}V_i
\]

## 13. Notebook outputs

Notebook 17 writes:

- `results/17_chunking/17_artifact_identity.json`
- `results/17_chunking/17_chunk_manifest.json`
- `results/17_chunking/17_chunk_table.csv`
- `results/17_chunking/17_chunk_verification.csv`
- `results/17_chunking/17_reassembly_report.json`
- `results/17_chunking/17_corrupt_chunk_report.csv`
- `results/17_chunking/17_merkle_preview.json`
- `figures/17_chunking_pipeline.png`
- `figures/17_chunk_manifest.png`
- `figures/17_chunk_verification.png`
- `figures/17_reassembly.png`
- `figures/17_corrupt_chunk_detection.png`
- `figures/17_merkle_preview.png`

In [ ]:
notebook_manifest = {
    "notebook": "17_chunking.ipynb",
    "title": "Chunking Specifies Large Model Distribution",
    "primary_specification": "chunking",
    "statement": "Large artifacts become distributable when split into independently verifiable transfer units.",
    "equations": [
        "C=(c_0,c_1,...,c_{n-1})",
        "I_i=H(c_i)",
        "M={(i,I_i,|c_i|)}",
        "V_i=(H(c_i_received)=I_i)",
        "R_assemble=AND_i V_i",
    ],
    "outputs": {
        "artifact_identity": str(artifact_record_path.relative_to(REPO_ROOT)),
        "chunk_manifest": str(chunk_manifest_path.relative_to(REPO_ROOT)),
        "chunk_table_csv": str(chunk_table_csv.relative_to(REPO_ROOT)),
        "chunk_verification_csv": str(verification_csv.relative_to(REPO_ROOT)),
        "reassembly_report": str(reassembly_report_path.relative_to(REPO_ROOT)),
        "corrupt_chunk_report": str(corrupt_report_csv.relative_to(REPO_ROOT)),
        "merkle_preview": str(merkle_preview_path.relative_to(REPO_ROOT)),
        "chunking_pipeline_figure": str(chunking_pipeline_fig.relative_to(REPO_ROOT)),
        "chunk_manifest_figure": str(chunk_manifest_fig.relative_to(REPO_ROOT)),
        "chunk_verification_figure": str(chunk_verification_fig.relative_to(REPO_ROOT)),
        "reassembly_figure": str(reassembly_fig.relative_to(REPO_ROOT)),
        "corrupt_chunk_figure": str(corrupt_chunk_fig.relative_to(REPO_ROOT)),
        "merkle_preview_figure": str(merkle_preview_fig.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "17_notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

notebook_manifest

## 14. Download artifacts

Run the final cell to package notebook 17 outputs for download.

In [ ]:
# Package notebook artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

NOTEBOOKS = REPO_ROOT / "notebooks"
SITE = REPO_ROOT / "site"

zip_path = RESULTS / "17_chunking_artifacts.zip"

notebook_path = NOTEBOOKS / "17_chunking.ipynb"
fallback_notebook_path = Path.cwd() / "17_chunking.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)

        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)

        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass